In [0]:
!pip install -qU torch
!pip install -q transformers==4.44.1
!pip install -q accelerate
!pip install -q bitsandbytes==0.43.3
!pip install -q datasets==2.21.0
!pip install -q trl==0.9.6
!pip install -q peft==0.12.0
!pip install -U "huggingface_hub[cli]"
dbutils.library.restartPython()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.18.1+cu121 requires torch==2.3.1, but you have torch 2.5.1 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
petastorm 0.12.1 requires pyspark>=2.1.0, which is not install

In [0]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

2025-01-21 17:27:38.797747: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-21 17:27:38.833430: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [0]:
hf_token = "<your-token-here>"

In [0]:
!huggingface-cli login --token "<your-token-here>" --add-to-git-credential

Token is valid (permission: write).
The token `diffusion` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved in your configured git credential helpers (cache).
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `diffusion`


### Load the model

In [0]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)

model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
quantized_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config = bnb_config,
                    device_map = "auto")
                    
tokenizer = AutoTokenizer.from_pretrained(model_name)
input = tokenizer("Natalia sold clips to 48 of her friends in April, and then she sold half as \
many clips in May. How many clips did Natalia sell altogether in April and May?", return_tensors="pt").to('cuda')

response = quantized_model.generate(**input, max_new_tokens = 100)
print(tokenizer.batch_decode(response, skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? A) 96 B) 100 C) 120 D) 144 E) 160\n## Step 1: Determine the number of clips sold in April.\nNatalia sold 48 clips in April.\n\n## Step 2: Calculate the number of clips sold in May.\nShe sold half as many clips in May, which means she sold 48 / 2 = 24 clips in May.\n\n## Step 3: Add the number of clips sold in April and May to']


In [0]:
param_dtypes = [param.dtype for param in quantized_model.parameters()]
print("Parameter dtypes:", param_dtypes)

Parameter dtypes: [torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.uint8, torch.float16, torch.float16

In [0]:
for name, module in quantized_model.named_modules():
    print(name)


model
model.embed_tokens
model.layers
model.layers.0
model.layers.0.self_attn
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.self_attn.rotary_emb
model.layers.0.mlp
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.act_fn
model.layers.0.input_layernorm
model.layers.0.post_attention_layernorm
model.layers.1
model.layers.1.self_attn
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.self_attn.rotary_emb
model.layers.1.mlp
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.1.mlp.act_fn
model.layers.1.input_layernorm
model.layers.1.post_attention_layernorm
model.layers.2
model.layers.2.self_attn
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.

In [0]:
print(quantized_model.model)  # The base transformer
print(quantized_model.model.layers[0])  # The first Transformer layer

LlamaModel(
  (embed_tokens): Embedding(128256, 4096)
  (layers): ModuleList(
    (0-31): 32 x LlamaDecoderLayer(
      (self_attn): LlamaSdpaAttention(
        (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((4096,), eps=1e-05)
  (r

In [0]:
layer = quantized_model.model.layers[0]
print(layer.self_attn)  # Self-attention module
print(layer.mlp)        # Feed-forward (MLP) module

LlamaSdpaAttention(
  (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
  (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
  (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (rotary_emb): LlamaRotaryEmbedding()
)
LlamaMLP(
  (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
  (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
  (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
  (act_fn): SiLU()
)


In [0]:
for name, param in quantized_model.named_parameters():
    print(name, param.shape)

model.embed_tokens.weight torch.Size([128256, 4096])
model.layers.0.self_attn.q_proj.weight torch.Size([8388608, 1])
model.layers.0.self_attn.k_proj.weight torch.Size([2097152, 1])
model.layers.0.self_attn.v_proj.weight torch.Size([2097152, 1])
model.layers.0.self_attn.o_proj.weight torch.Size([8388608, 1])
model.layers.0.mlp.gate_proj.weight torch.Size([29360128, 1])
model.layers.0.mlp.up_proj.weight torch.Size([29360128, 1])
model.layers.0.mlp.down_proj.weight torch.Size([29360128, 1])
model.layers.0.input_layernorm.weight torch.Size([4096])
model.layers.0.post_attention_layernorm.weight torch.Size([4096])
model.layers.1.self_attn.q_proj.weight torch.Size([8388608, 1])
model.layers.1.self_attn.k_proj.weight torch.Size([2097152, 1])
model.layers.1.self_attn.v_proj.weight torch.Size([2097152, 1])
model.layers.1.self_attn.o_proj.weight torch.Size([8388608, 1])
model.layers.1.mlp.gate_proj.weight torch.Size([29360128, 1])
model.layers.1.mlp.up_proj.weight torch.Size([29360128, 1])
model.

### Preprocess the dataset

In [0]:
from datasets import load_dataset
from datasets import Dataset
import pandas as pd

dataset = "openai/gsm8k"
# data = load_dataset(dataset, 'main')
data = pd.read_parquet("data/gsm8k/train-00000-of-00001.parquet")

data = data.sample(400, random_state=42).reset_index(drop=True)
print(data.shape)
data = Dataset.from_pandas(data)

(400, 2)


In [0]:
tokenizer.pad_token = tokenizer.eos_token
train_sample = data.map(lambda row: tokenizer(row["question"], row["answer"], truncation=True, padding="max_length", max_length=100), batched=True)
display(train_sample)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask'],
    num_rows: 400
})

### LoRA configurations

In [0]:
import peft
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

In [0]:
# TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING = {
#     "t5": ["q", "v"],
#     "mt5": ["q", "v"],
#     "bart": ["q_proj", "v_proj"],
#     "gpt2": ["c_attn"],
#     "bloom": ["query_key_value"],
#     "blip-2": ["q", "v", "q_proj", "v_proj"],
#     "opt": ["q_proj", "v_proj"],
#     "gptj": ["q_proj", "v_proj"],
#     "gpt_neox": ["query_key_value"],
#     "gpt_neo": ["q_proj", "v_proj"],
#     "bert": ["query", "value"],
#     "roberta": ["query", "value"],
#     "xlm-roberta": ["query", "value"],
#     "electra": ["query", "value"],
#     "deberta-v2": ["query_proj", "value_proj"],
#     "deberta": ["in_proj"],
#     "layoutlm": ["query", "value"],
#     "llama": ["q_proj", "v_proj"],
#     "chatglm": ["query_key_value"],
#     "gpt_bigcode": ["c_attn"],
#     "mpt": ["Wqkv"],
# }

### Training arguments

In [0]:
from transformers import TrainingArguments
import os

working_dir = 'models/'
output_directory = os.path.join(working_dir, "lora")

training_args = TrainingArguments(
    output_dir = output_directory,
    auto_find_batch_size = True,
    learning_rate = 3e-4,
    num_train_epochs=2
)

### Set the trainer

In [0]:
import transformers
from trl import SFTTrainer

trainer = SFTTrainer(
    model = quantized_model,
    args = training_args,
    train_dataset = train_sample,
    peft_config = lora_config,
    tokenizer = tokenizer,
    data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

[2025-01-21 17:27:55,803] [INFO] [real_accelerator.py:203:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


 [WARNING]  async_io requires the dev libaio .so object and headers but these were not found.
 [WARNING]  async_io: please install the libaio-dev package with apt
 [WARNING]  If libaio is already installed (perhaps from source), try setting the CFLAGS and LDFLAGS environment variables to where it can be found.
 [WARNING]  Please specify the CUTLASS repo directory as environment variable $CUTLASS_PATH
 [WARNING]  sparse_attn requires a torch version >= 1.5 and < 2.0 but detected 2.5
 [WARNING]  using untested triton version (3.1.0), only 1.0.0 is known to be compatible


/databricks/python/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @autocast_custom_fwd
/databricks/python/lib/python3.11/site-packages/deepspeed/runtime/zero/linear.py:66: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @autocast_custom_bwd
/local_disk0/.ephemeral_nfs/envs/pythonEnv-fd5188ab-bca4-4265-82a8-dcf0df051f39/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:289: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
[rank0]:[W121 17:27:58.803722181 ProcessGroupNCCL.cpp:4115] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 to perform barrier as devices used by this process are currently unknown. This can potentially cause a hang if this rank to GPU mapping is incorre

In [0]:
trainer.train()

[rank0]:[W121 17:28:01.153344477 reducer.cpp:1400] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())


Step,Training Loss


TrainOutput(global_step=100, training_loss=1.4040701293945312, metrics={'train_runtime': 53.7265, 'train_samples_per_second': 14.89, 'train_steps_per_second': 1.861, 'total_flos': 3605635513974784.0, 'train_loss': 1.4040701293945312, 'epoch': 2.0})

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


### Save the model

In [0]:
# Save the model.
model_path = os.path.join(output_directory, f"qlora_model")
trainer.model.save_pretrained(model_path)

### Load the fine-tuned model

In [0]:
model_path = "/Workspace/Users/lep5kor@bosch.com/models/lora/qlora_model"

from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
)

loaded_model = AutoPeftModelForCausalLM.from_pretrained(
                                        model_path,
                                        quantization_config = bnb_config,
                                        device_map = 'auto')
                                        
tokenizer = AutoTokenizer.from_pretrained(model_name)
input = tokenizer("Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?", return_tensors="pt").to('cuda')

response = loaded_model.generate(**input, max_new_tokens = 100)
print(tokenizer.batch_decode(response, skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-fd5188ab-bca4-4265-82a8-dcf0df051f39/lib/python3.11/site-packages/bitsandbytes/nn/modules.py:435: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?Question: Natalia sold clips to 48 friends in April. In May, she sold half as many clips as in April, so she sold 48/2= <<48/2=24>>24 clips in May.\nIn total, Natalia sold 48+24= <<48+24=72>>72 clips in April and May.\n#### 72 clips were sold in total. ####\n### 72 clips were sold in total. ###\n#### 72 clips were sold in']
